# 🎨 ComfyUI Pro + Test de LoRAs de **Claire (Lost Sword)** en Google Colab

Este notebook está configurado para ejecutar **ComfyUI** en Google Colab, cargando automáticamente el modelo base **Illustrious-XL** y todos los **checkpoints de LoRA de Claire** que acabas de entrenar (épocas 2, 4, 6 y 8).

### ✨ Incluye:
- ComfyUI + ComfyUI Manager + Impact Pack + IP-Adapter Plus + AnimateDiff + Wan 2.1
- Sincronización automática de LoRAs desde tu Google Drive (`/MyDrive/Illustrious_LoRAs/`) o de la sesión actual
- Túnel de Cloudflare para acceso instantáneo a la interfaz gráfica.

In [ ]:
# 1️⃣ INSTALACIÓN BASE DE COMFYUI + CUSTOM NODES + GOOGLE DRIVE
import os, sys, subprocess

# 1. Actualizar e instalar dependencias del sistema
!apt -y update -qq
!apt -y install aria2 ffmpeg git -qq

# 2. Clonar ComfyUI Base si no existe
if not os.path.exists('/content/ComfyUI'):
    print('🚀 Clonando ComfyUI...')
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121 -q
!pip install -r requirements.txt -q

# 3. Instalación de Nodos Personalizados (Custom Nodes)
CUSTOM_NODES = '/content/ComfyUI/custom_nodes'
os.makedirs(CUSTOM_NODES, exist_ok=True)

nodes = {
    'ComfyUI-Manager': 'https://github.com/ltdrdata/ComfyUI-Manager',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite',
    'ComfyUI-Impact-Pack': 'https://github.com/ltdrdata/ComfyUI-Impact-Pack',
    'ComfyUI-WD14-Tagger': 'https://github.com/pythongosssss/ComfyUI-WD14-Tagger',
    'ComfyUI-AnimateDiff-Evolved': 'https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved',
    'ComfyUI_FizzNodes': 'https://github.com/FizzleDorf/ComfyUI_FizzNodes',
    'ComfyUI-WanVideoWrapper': 'https://github.com/Kijai/ComfyUI-WanVideoWrapper',
    'ComfyUI_IPAdapter_plus': 'https://github.com/cubiq/ComfyUI_IPAdapter_plus',
    'ComfyUI_Style_Aligned': 'https://github.com/leeguandong/ComfyUI_Style_Aligned'
}

for name, url in nodes.items():
    path = os.path.join(CUSTOM_NODES, name)
    if not os.path.exists(path):
        print(f'📦 Instalando {name}...')
        !git clone {url} {path}
        req_file = os.path.join(path, 'requirements.txt')
        if os.path.exists(req_file):
            !pip install -r {req_file} -q

# 4. Conectar Google Drive para guardar renders y leer LoRAs
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
os.makedirs('/content/drive/MyDrive/ComfyUI/output', exist_ok=True)
!rm -rf /content/ComfyUI/output
!ln -sf /content/drive/MyDrive/ComfyUI/output /content/ComfyUI/output

print('\n✅ Celda 1 Completada: ComfyUI, Custom Nodes y Google Drive listos.')

In [ ]:
# 2️⃣ DESCARGAR MODELO ILLUSTRIOUS + IMPORTAR LoRAs DE CLAIRE (TODAS LAS ÉPOCAS)
import requests, os, shutil, glob
from huggingface_hub import hf_hub_download

CIVITAI_TOKEN = "1b82d9ca1d1220d1e15a6ea0d1aafb97"

MODEL_DIR = "/content/ComfyUI/models/checkpoints"
LORA_DIR = "/content/ComfyUI/models/loras"
VAE_DIR = "/content/ComfyUI/models/vae"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LORA_DIR, exist_ok=True)
os.makedirs(VAE_DIR, exist_ok=True)

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

def download_civitai(version_id, dest_dir, filename, token):
    dest = os.path.join(dest_dir, filename)
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        mb = os.path.getsize(dest) // 1_048_576
        print(f'ℹ️ {filename} ({mb} MB) ya existe')
        return
    url = f'https://civitai.com/api/download/models/{version_id}'
    r = requests.get(url, params={'token': token}, headers=headers, stream=True)
    if r.status_code == 200:
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
        print(f'✅ Descargado: {filename}')
    else:
        print(f'❌ Error {r.status_code} al descargar {filename}')

# ── A. Modelo Base Illustrious-XL v0.1 ──
IL_CHECKPOINT = os.path.join(MODEL_DIR, "Illustrious-XL-v0.1.safetensors")
if os.path.exists("/content/models/Illustrious-XL-v0.1.safetensors"):
    if not os.path.exists(IL_CHECKPOINT):
        !ln -sf /content/models/Illustrious-XL-v0.1.safetensors {IL_CHECKPOINT}
    print("✅ Illustrious-XL enlazado directamente desde la sesión de entrenamiento.")
elif not os.path.exists(IL_CHECKPOINT):
    print("⬇️ Descargando Illustrious-XL v0.1...")
    hf_hub_download(
        repo_id="OnomaAIResearch/Illustrious-xl-early-release-v0",
        filename="Illustrious-XL-v0.1.safetensors",
        local_dir=MODEL_DIR
    )
    print("✅ Illustrious-XL descargado.")
else:
    print("ℹ️ Illustrious-XL ya existe en checkpoints.")

# ── B. VAE SDXL ──
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors -d {VAE_DIR} -o sdxl_vae.safetensors

# ── C. IMPORTAR TODOS LOS LORAS DE CLAIRE (Épocas 2, 4, 6, 8, final) ──
print("\n🔍 Buscando LoRAs entrenados de Claire...")
# 1. Desde la carpeta de salida del entrenamiento
for f in glob.glob("/content/output_lora/*.safetensors"):
    shutil.copy(f, LORA_DIR)
    print(f"  ✓ LoRA importado desde output: {os.path.basename(f)}")
# 2. Desde Google Drive
for f in glob.glob("/content/drive/MyDrive/Illustrious_LoRAs/*.safetensors"):
    shutil.copy(f, LORA_DIR)
    print(f"  ✓ LoRA importado desde Drive: {os.path.basename(f)}")

# ── D. LoRAs de Calidad y Detalle (Civitai) ──
print("\n🎨 Descargando LoRAs de utilidad y detalle...")
download_civitai(1486887, LORA_DIR, 'add-detail-xl.safetensors', CIVITAI_TOKEN)       # Add Detail XL
download_civitai(2769575, LORA_DIR, 'Face_Detailed_v2.safetensors', CIVITAI_TOKEN)    # Face Detail
download_civitai(1444863, LORA_DIR, 'darkness_slider.safetensors', CIVITAI_TOKEN)     # Darkness Slider
download_civitai(2073647, LORA_DIR, 'stabilizer_animaginexl.safetensors', CIVITAI_TOKEN) # Stabilizer

print(f'\n✅ Celda 2 Completada: Todos los modelos y LoRAs de Claire están en {LORA_DIR}.')

In [ ]:
# 3️⃣ DESCARGA DE COMPONENTES DE VIDEO E IP-ADAPTER
import os

AD_MODELS = '/content/ComfyUI/models/animatediff_models'
AD_EVOLVED_MODELS = '/content/ComfyUI/custom_nodes/ComfyUI-AnimateDiff-Evolved/models'
DIFFUSION_DIR = '/content/ComfyUI/models/diffusion_models'
CLIP_DIR = '/content/ComfyUI/models/clip'
CLIP_VISION_DIR = '/content/ComfyUI/models/clip_vision'
IPADAPTER_DIR = '/content/ComfyUI/models/ipadapter'
LORA_DIR = '/content/ComfyUI/models/loras'

for d in [AD_MODELS, AD_EVOLVED_MODELS, DIFFUSION_DIR, CLIP_DIR, CLIP_VISION_DIR, IPADAPTER_DIR, LORA_DIR]:
    os.makedirs(d, exist_ok=True)

# A. Modelos IP-Adapter Plus SDXL + CLIP Vision (ViT-H)
print('🎨 Descargando CLIP-Vision Encoder (ViT-H)...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors -d {CLIP_VISION_DIR} -o CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors

print('🎨 Descargando IP-Adapter Plus SDXL...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors -d {IPADAPTER_DIR} -o ip-adapter-plus_sdxl_vit-h.safetensors

# B. AnimateDiff Motion Model SDXL
print('🎬 Descargando AnimateDiff Motion Model SDXL...')
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/guoyww/animatediff/resolve/main/mm_sdxl_v10_beta.ckpt -d {AD_MODELS} -o mm_sdxl_v10_beta.ckpt
!cp -f {AD_MODELS}/mm_sdxl_v10_beta.ckpt {AD_EVOLVED_MODELS}/mm_sdxl_v10_beta.ckpt 2>/dev/null || true

print('\n✅ Celda 3 Completada: AnimateDiff e IP-Adapter listos.')

In [ ]:
# 4️⃣ INICIAR COMFYUI + ENLACE PÚBLICO (CLOUDFLARE)
import threading, time

%cd /content/ComfyUI

# Iniciar servidor de ComfyUI en segundo plano
def run_comfyui():
    !python main.py --listen 0.0.0.0 --port 8188 --enable-manager --highvram

t = threading.Thread(target=run_comfyui)
t.start()
time.sleep(5)

# Iniciar túnel de Cloudflare para obtener enlace público de acceso
print('\n🌐 Generando enlace público de acceso a ComfyUI...')
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!./cloudflared tunnel --url http://127.0.0.1:8188